# Select and Filter

The product team at Wavelength needs answers. Which tracks are trending?
Who are the top listeners? What genres dominate the platform?

You will answer these questions with SQL -- the language that every data
engineer reaches for first. If data is stored in a table (and it usually is),
SQL is how you get it out.

## Setup

We need a small amount of plumbing before we can start writing queries.
First, suppress some harmless but distracting warnings from `jupysql`.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

Install `jupysql` and load the SQL extension, then connect to the Wavelength
database.

In [ ]:
%pip install -q jupysql
%load_ext sql

In [ ]:
%sql sqlite:///../data/wavelength.sqlite

## First look at the data

The best way to understand a database is to look at it. The `*` means
"every column".

In [ ]:
%sql SELECT * FROM tracks LIMIT 10;

## Exploring the database

The `tracks` table is one of several. `%sqlcmd tables` lists them all, and
`%sqlcmd columns` shows the structure of a specific table.

In [ ]:
%sqlcmd tables

In [ ]:
%sqlcmd columns --table tracks

Notice that `play_count` is nullable -- some tracks have no play count
recorded. We will come back to that, because NULLs in SQL behave in ways
that catch everyone out.

## Selecting specific columns and aliases

Grabbing every column with `*` is fine for exploration, but in production
you should only select what you need. It is faster, and it makes your
queries self-documenting. Use `AS` to create aliases that give columns
friendlier names in the output.

In [ ]:
%%sql
SELECT
    title AS track_name,
    duration_seconds AS length_secs,
    play_count AS plays
FROM tracks
LIMIT 10;

## Filtering with WHERE

Rarely do you want every row. The `WHERE` clause filters down to rows
matching a condition. Text values go in single quotes; the comparison
operators are `=`, `!=` (or `<>`), `<`, `>`, `<=`, `>=`.

In [ ]:
%%sql
SELECT title, genre, play_count
FROM tracks
WHERE genre = 'rock'
LIMIT 10;

In [ ]:
%%sql
SELECT title, genre, duration_seconds
FROM tracks
WHERE duration_seconds > 300
LIMIT 10;

## Combining conditions: AND / OR

Real questions usually involve more than one condition. AND narrows; OR
broadens. When mixing them, use brackets to make your intent clear --
without them, SQL may interpret the logic differently than you expect.

In [ ]:
%%sql
SELECT title, genre, play_count
FROM tracks
WHERE (genre = 'rock' OR genre = 'jazz')
  AND play_count > 5000
LIMIT 10;

## Pattern matching with LIKE

`LIKE` lets you use wildcards: `%` matches any sequence of characters,
`_` matches exactly one character.

In [ ]:
%%sql
SELECT title, genre
FROM tracks
WHERE title LIKE '%Fire%'
LIMIT 10;

## Sorting and limiting: ORDER BY and LIMIT

`ORDER BY` sorts results (`DESC` for descending, `ASC` for ascending).
`LIMIT` caps the number of rows returned. You can sort by multiple columns.

In [ ]:
%%sql
SELECT title, genre, play_count
FROM tracks
WHERE play_count IS NOT NULL
ORDER BY genre ASC, play_count DESC
LIMIT 15;

## NULL handling: the trap everyone falls into

This is possibly the most important concept in this entire notebook, and it
trips up experienced engineers, not just beginners.

Some tracks have no recorded `play_count`. Let's try to find them.

In [ ]:
%%sql
SELECT title, play_count
FROM tracks
WHERE play_count = NULL
LIMIT 10;

Nothing. Zero rows. But we know there are tracks without a play count.

Here is the key insight: **NULL does not equal anything, not even itself.**
`NULL = NULL` returns NULL, not TRUE. In a WHERE clause, NULL is treated as
"not true", so the row is excluded.

Think of NULL as meaning "unknown". Is this unknown value equal to that
unknown value? The honest answer is "I don't know" -- which is NULL.
Unknown compared to unknown is still unknown.

A track with no play count is not the same as a track with zero plays.
Zero is a known quantity. NULL means the data simply is not there.

The correct way to check for NULL is `IS NULL` (or `IS NOT NULL`).

In [ ]:
%%sql
SELECT title, play_count
FROM tracks
WHERE play_count IS NULL
LIMIT 10;

If you are building a dashboard that shows "tracks with fewer than 100
plays", `WHERE play_count < 100` will silently exclude all NULL tracks.
Those tracks might have millions of plays, or none. You do not know. That
is exactly what NULL means.

## Putting it together

Let's answer the product team's question: "Top 20 trending rock and pop
tracks?" -- highest play count, excluding unknowns.

In [ ]:
%%sql
SELECT
    title AS track_name,
    genre,
    play_count AS plays
FROM tracks
WHERE (genre = 'rock' OR genre = 'pop')
  AND play_count IS NOT NULL
ORDER BY play_count DESC
LIMIT 20;

That query uses column selection, aliases, filtering, compound conditions,
NULL handling, sorting, and limiting.

## Exercises

Write your SQL in the empty code cells below.

### Exercise 1

Select the `username`, `country`, and `subscription_type` from the `users`
table. Limit to 10 rows.

### Exercise 2

Find all artists from the UK. Select their `name`, `genre`, and
`formed_year`.

### Exercise 3

Find all electronic tracks with a play count greater than 5,000. Show the
title and play count, sorted by play count descending.

### Exercise 4

How many tracks have a NULL play count? Write a query that returns all
columns for those tracks. Remember, `= NULL` does not work.

### Exercise 5

Find all premium users who signed up in 2024 or later. Show their username,
signup date, and subscription type, sorted by signup date.

**Hint:** Dates are stored as text in `YYYY-MM-DD` format, so string
comparison works: `signup_date >= '2024-01-01'`.

## Summary

- **SELECT** chooses columns; **FROM** specifies the table
- **WHERE** filters rows by condition
- **AND / OR** combine conditions (use brackets to be explicit)
- **LIKE** matches text patterns with `%` and `_` wildcards
- **ORDER BY** sorts results; **LIMIT** caps the count
- **NULL** is not a value -- it is the absence of one. Use `IS NULL` and
  `IS NOT NULL`, never `= NULL`

In the next notebook, we start combining data from multiple tables using
joins.